In [ ]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

import hces_utils
from hces_utils import weighted_avg_and_std, weighted_summary

!date

# HCES Data Extraction
We have HCES data to inform salt consumption for all household members in India, by sex, age group (0-5, 5-15, 15-30, 30-50, 50+)
and wealth quintile. HCES does not record pregnancy status, so pregnant women get the values for their sex, age group and quintile.

Old HCES data and documentation is saved on Sharepoint here: https://uwnetid.sharepoint.com/:f:/r/sites/ihme_simulation_science_team/Shared%20Documents/Research/LSFF/07_Data/HCES_22_data_files?csf=1&web=1&e=wmxQSD; new docs on HCES 2023-24 are stored in
`/snfs1/Project/simulation_science/IND_HOUSEHOLD_CONSUMPTION_EXPENDITURE_SURVEY_HCES_2023_2024/`

Item codes for 'Salt' include: 073, 178, and 170. See Section 5.3 of the above documentation for more context.
- Item code 178 signifies salt procured through PDS, using ration card.
- Item code 073 signifies salt procured through PDS, free of charge.
- Item code 170 signifies salt procured/consumed from other sources.

We also will need to approximate DHS wealth quintiles in the HCES data, by using similar variables as is used by the DHS to 
calculate wealth quintiles in India (see DHS wealth quintile documentation here: 
https://dhsprogram.com/programming/wealth%20index/India%20DHS%202015-16/India%202015-16%20sps.txt)

In this notebook, we extract the raw data from the HCES website and do some initial processing in order to create files 
that we can further tabulate based on our project needs in a later step. For now, we process these raw data into dataframes 
that have a row for each household/individual with the following columns:
- Whether or not salt was consumed (binary variable - this will be used to calculate coverage percentages later)
- Amount of salt consumed (grams per day - currently in raw HCES as kg/30 days for household) 
- Wealth composite score/quintile 
- Sex/gender (will be used to tabulate by sex)
- Age (will be used to tabulate age groups)



## Load data

In [ ]:
households, household_members_raw = hces_utils.load_households()
households

In [ ]:
# The fact that there are a couple very large households (e.g. 20+ members) is a little
# surprising - maybe we can investigate to confirm these are some kind of GQ.
households.hh_size.hist()

In [ ]:
# Bizarrely, I can't find documentation of these codes -- but we know the rent rate is only asked for rural
households.assign(rent_rate=households.rural_rent_rate_availability.notnull()).groupby(
    "sector"
).rent_rate.mean()

In [ ]:
households.sector.value_counts(dropna=False)

In [ ]:
households.isnull().mean().sort_values()

## Calculate wealth quintiles

In order to add wealth stratifications into our HCES data, we will emulate DHS methods for calculating wealth quintiles.
They don't ask something like "how wealthy are you?"
Instead, they ask "do you have a refrigerator?" and then a bunch of other questions like that,
and analyze them into a composite wealth index.
The way they do so is actually pretty weird and interesting, but I won't get into that here.
It is described at https://dhsprogram.com/topics/wealth-index/Wealth-Index-Construction.cfm.

In a first pass, we attempted to use the DHS weights directly.
See [this past version of the notebook](https://github.com/ihmeuw/vivarium_gates_lsff_2026_maternal/blob/12d63b4c5a3b7cb204c4f6eb3de7d6ffe87ce9c3/0100_data_prep/01_extract_hces.ipynb)
for how we started to do this.

However, this was tricky,
because none of the questions asked in HCES are exactly the same, with the same answer options,
as in DHS.
We could sort of use proxies and fudge things, but it was pretty rough.
Here, I have instead used the same *methods* used in the DHS to make an entirely new
model of the latent wealth variable.
I followed the steps described here: https://dhsprogram.com/programming/wealth%20index/Steps_to_constructing_the_new_DHS_Wealth_Index.pdf

However, even this is pretty underwhelming. Probably a better strategy to try in the future would be to make a more complex and
precise crosswalk between DHS and HCES questions/responses, then train a model in DHS to predict the DHS wealth quintile based
only on information available in HCES, and finally predict HCES.

DHS wealth index variables include: 
- Source of drinking water
- Type of toilet facility
- Electricity
- Mattress
- Pressure cooker
- Chair
- Cot or bed
- Table
- Electric fan
- Radio or transistor
- Black and white television
- Colour television
- Sewing machine
- Mobile telephone
- Telephone (non-mobile)
- Internet
- Computer
- Refrigerator
- Air conditioner/cooler
- Washing machine
- Watch or clock
- Bicycle
- Motorcycle or Scooter
- Animal-drawn cart
- Car
- Water pump
- Thresher
- Tractor
- Type of cooking fuel
- Main material of floor
- Main roof material
- Main wall material
- Hectares for agricultural land
- Out of this land, how much is irrigated?
- Cows / bulls / buffaloes
- Camels
- Horses / donkeys / mules
- Goats
- Sheep
- Chickens / ducks
- Bank account
- Members per sleeping room
- Compute urban and rural variables coded (1/0) for filters later
- Toilet facility by shared/not shared
- Land area by units - if there are separate units - need to convert them to one unit

HCES variables:
- Type of land owned
- What is the total area of all owned (owned and possessed or leased out) land (within the country) by the household as on the date of survey (area in acre)?(upto two places of decimal) 
- Primary source of energy of the household for cooking
- Any household member used the internet in the last 30 days (from the person-level level 2 records)
- Whether household possessed one or more item as on the date of the survey- Television 
- Whether household possessed one or more item as on the date of the survey- Radio
- Whether household possessed one or more item as on the date of the survey - Laptop/PC
- Whether household possessed one or more item as on the date of the survey- Mobile handset
- Whether household possessed one or more item as on the date of the survey- Bicycle
- Whether household possessed one or more item as on the date of the survey- Motorcycle, scooter 
- Whether household possessed one or more item as on the date of the survey- Motor car/jeep/van
- Whether household possessed one or more item as on the date of the survey- Trucks
- Whether household possessed one or more item as on the date of the survey - Animal cart
- Whether household possessed one or more item as on the date of the survey- Refrigerator
- Whether household possessed one or more item as on the date of the survey- Washing machine
- Whether household possessed one or more item as on the date of the survey- Air conditioner/air cooler 
- Type of multichannel television facility is used by the household as on the date of the survey

Crosswalk spreadsheet saved here: https://uwnetid.sharepoint.com/sites/ihme_simulation_science_team/_layouts/15/doc.aspx?sourcedoc={eddcbf51-3462-47d7-b847-79edea5046c6}&action=edit


In [ ]:
wealth_info, wealth_predictors = hces_utils.assign_wealth_quintiles(
    households, diagnostics=True
)
wealth_info

In [ ]:
wealth_info.combined_score.hist(bins=500)

In [ ]:
wealth_info.groupby(
    "wealth_quintile", observed=True
).multiplier.sum() / wealth_info.multiplier.sum()

In [ ]:
for column in wealth_predictors:
    print(column)
    print(
        wealth_info.groupby(
            ["wealth_quintile", column], observed=True
        ).multiplier.sum()
        / wealth_info.groupby("wealth_quintile").multiplier.sum()
    )

In [ ]:
# Whoa, owning land or a house is uncorrelated?!
india_dhs = pd.read_stata(
    "/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/IND_DHS7_2015_2016_HH_IAHR74FL_Y2018M12D06.DTA",
    columns=["hv005", "hv270", "sh46"],
).rename(columns={"hv005": "weight", "hv270": "wealth_quintile", "sh46": "owns_house"})

In [ ]:
india_dhs.groupby(
    ["wealth_quintile", "owns_house"], observed=True
).weight.sum() / india_dhs.groupby("wealth_quintile").weight.sum()

In [ ]:
# Bizarre. Even in DHS it's basically uncorrelated, although in HCES it seems to be reversed.

In [ ]:
households["wealth_quintile"] = wealth_info["wealth_quintile"]

In [ ]:
assert households.wealth_quintile.notnull().all()

## Household salt consumption

Item codes for 'Salt' include: 073, 178, and 170. See Section 5.3 of the above documentation for more context.

- Item code 178 signifies salt procured through PDS, using ration card.
- Item code 073 signifies salt procured through PDS, free of charge.
- Item code 170 signifies salt procured/consumed from other sources.

Units: kg/30 days.

In [ ]:
consumption = hces_utils.read_consumption([73, 170, 178])
consumption

In [ ]:
consumption["item_code"] = consumption.item_code.map(
    {
        73: "pds free salt",
        178: "pds paid salt",
        170: "other salt",
    }
)

In [ ]:
consumption["source"] = consumption.source.map(
    {
        1: "only purchase",
        2: "only home-grown",
        3: "both purchase and home-grown",
        4: "only free collection",
        5: "only exchange",
        6: "only gifts",
        7: "received free through PDS",
        9: "other",
    }
)

In [ ]:
consumption["total_consumption_amt"] = consumption.total_consumption_amt.astype(float)
consumption["consumption_amt_home_produce"] = (
    consumption.consumption_amt_home_produce.astype(float)
)

In [ ]:
# I assume for these high-missingness columns, null means 0
consumption.isnull().mean()

In [ ]:
(consumption == 0).mean()

In [ ]:
consumption = consumption.fillna(0)

In [ ]:
consumption["salt"] = consumption.total_consumption_amt

In [ ]:
is_pds = consumption.item_code.isin(["pds free salt", "pds paid salt"])
consumption["salt_purchased"] = np.where(
    is_pds,
    # All PDS salt counts as purchased
    consumption.total_consumption_amt,
    np.where(
        # 170 (other sources): only the bought part
        consumption.source.isin(["only purchase", "both purchase and home-grown"]),
        consumption.total_consumption_amt - consumption.consumption_amt_home_produce,
        0.0,
    ),
)

In [ ]:
consumption["salt_non_purchased"] = consumption.salt - consumption.salt_purchased

In [ ]:
consumption = consumption.groupby("common_id")[
    ["salt", "salt_purchased", "salt_non_purchased"]
].sum()
consumption

In [ ]:
consumption["proportion_purchased"] = (
    consumption.salt_purchased / consumption.salt
).clip(0, 1)
consumption

In [ ]:
len(consumption) / len(households)

In [ ]:
(households.merge(consumption, on="common_id", how="left").salt > 0).mean()

In [ ]:
consumption = consumption.reindex(households.common_id).fillna(0).reset_index()
consumption

In [ ]:
(households.merge(consumption, on="common_id", how="left").salt > 0).mean()

## Individual salt consumption
Each person's salt consumption is then their total meals (at home and away from home: school/balwadi, employer, paid, and other meals) multiplied by their household's per-meal amount. That is, we assume meals eaten away from home contain salt at the same rate as the household's meals at home.

In [ ]:
# Level 2 was read once with the households (for the internet indicator); reuse it.
household_members = hces_utils.load_household_members(household_members_raw, households)
household_members

In [ ]:
household_members = hces_utils.add_total_meals(household_members)

In [ ]:
household_members.plot(kind="scatter", x="usual_daily_meals", y="meals", alpha=0.01)

In [ ]:
household_meals = household_members.groupby(
    "common_id"
).meals_at_home_last_30_days.sum()
household_meals

In [ ]:
consumption = consumption.merge(
    household_meals, on="common_id", validate="1:1", how="left"
)
consumption

In [ ]:
consumption.isnull().mean()

In [ ]:
consumption = consumption.dropna(how="any").copy()
consumption

In [ ]:
consumption["salt_per_meal_kg"] = np.where(
    consumption.salt == 0,
    0.0,
    consumption.salt / consumption.meals_at_home_last_30_days,
)

In [ ]:
# In a small number of cases, salt consumption is reported despite very few or no meals at home
impossible_salt_consumption = (consumption.salt > 0) & (
    consumption.meals_at_home_last_30_days == 0
)
consumption[impossible_salt_consumption]

In [ ]:
# This must be wrong somehow, so we clip it
consumption["salt_per_meal_kg"] = consumption["salt_per_meal_kg"].clip(
    upper=np.percentile(consumption.salt_per_meal_kg, 95)
)

In [ ]:
consumption[consumption.salt_per_meal_kg.notnull()].sort_values("salt_per_meal_kg")
consumption.salt_per_meal_kg.hist()

In [ ]:
household_members = household_members.merge(
    consumption.reset_index()[
        [
            "common_id",
            "salt_per_meal_kg",
            "proportion_purchased",
        ]
    ],
    on="common_id",
    how="left",
    validate="m:1",
)
household_members

In [ ]:
household_members.isnull().mean()

In [ ]:
household_members["salt"] = household_members.meals * household_members.salt_per_meal_kg
assert (household_members.meals >= household_members.meals_at_home_last_30_days).all()

In [ ]:
household_members.isnull().mean()

In [ ]:
# Convert kg/30 days to g/day
household_members["salt_g_per_day"] = household_members.salt / 30 * 1_000

In [ ]:
def save_data(series, name):
    hces_utils.save_data(series, name, vehicle="salt")

In [ ]:
weighted_avg_and_std(
    household_members.salt_g_per_day > 0, weights=household_members.multiplier
)

In [ ]:
weighted_summary(
    household_members, lambda df: df.salt_g_per_day > 0, by="wealth_quintile"
)

In [ ]:
# NOTE: We do not have pregnancy status in this survey, so we assume it has no effect on consumption!
any_consumption = weighted_summary(
    household_members,
    lambda df: np.where(df.salt_g_per_day.notnull(), df.salt_g_per_day > 0, np.nan),
)
any_consumption

In [ ]:
save_data(any_consumption["mean"], "salt/vehicle_consumption/any")

In [ ]:
weighted_avg_and_std(
    household_members.salt_g_per_day, weights=household_members.multiplier
)

In [ ]:
weighted_summary(household_members, lambda df: df.salt_g_per_day, by="wealth_quintile")

In [ ]:
consumption_amount = weighted_summary(household_members, lambda df: df.salt_g_per_day)
consumption_amount

In [ ]:
save_data(consumption_amount["mean"], "salt/vehicle_consumption/amount/mean")
save_data(consumption_amount["sd"], "salt/vehicle_consumption/amount/sd")

In [ ]:
weighted_summary(
    household_members,
    lambda df: df.proportion_purchased,
    weights=lambda df: df.multiplier * df.salt,
    by="wealth_quintile",
)

In [ ]:
proportion_purchased = weighted_summary(
    household_members,
    lambda df: df.proportion_purchased,
    weights=lambda df: df.multiplier * df.salt,
)
proportion_purchased

In [ ]:
proportion_purchased["mean"].rename("value").to_csv(
    "india_salt_proportion_purchased.csv"
)